This notebook generates one mosaic from all the DTM files. 

In [ ]:
!pip install geopandas shapely rasterio

In [ ]:
import os
import boto3
import rasterio
import tempfile
from rasterio.merge import merge

In [ ]:
s3 = boto3.client(
    's3',
    endpoint_url=os.environ['S3_ENDPOINT_URL'],
    aws_access_key_id=os.environ['AWS_ACCESS_KEY_ID'],
    aws_secret_access_key=os.environ['AWS_SECRET_ACCESS_KEY']
)

bucket_name = 'dataspace'
path_dtm = 'city-data/dtm/latest/'

In [ ]:
output_path = '../../../datos-notebooks/1.HR-data-bologna/1.3.topography/dtm_mosaic.tif'

# Create a temporary directory on disk (avoids using RAM)
with tempfile.TemporaryDirectory() as tmpdir:
    # List all .ASC files in the given path
    response = s3.list_objects_v2(Bucket=bucket_name, Prefix=path_dtm)
    asc_files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.ASC')]

    local_files = []
    for key in asc_files:
        local_path = os.path.join(tmpdir, os.path.basename(key))
        # Download each file to the temp folder
        s3.download_file(bucket_name, key, local_path)
        local_files.append(local_path)

    # Open all rasters as datasets (but not all loaded in RAM)
    src_files_to_mosaic = [rasterio.open(fp) for fp in local_files]

    # Merge rasters on disk
    mosaic, out_trans = merge(src_files_to_mosaic)

    # Copy metadata from the first raster
    out_meta = src_files_to_mosaic[0].meta.copy()
    out_meta.update({
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_trans
    })

    # Write the final mosaic to disk (not in memory)
    with rasterio.open(output_path, "w", **out_meta) as dest:
        dest.write(mosaic)

    # Close datasets
    for src in src_files_to_mosaic:
        src.close()

print(f"Mosaic saved inn: {output_path}")

